# Nifty 50 IT Sector Time-Series Forecasting (Phase 3)

This analysis focuses on optimizing the predictive accuracy of the Indian IT sector benchmarks through the integration of advanced technical indicators and multivariate deep learning architectures. The primary target variable is the **Aggregate Performance Score (APS)**, a volume-weighted composite metric enhanced with temporal momentum and trend-continuation features.

### Pipeline Architecture
1. **Data Acquisition**: Automated retrieval of sector constituents and global macroeconomic benchmarks.
2. **Exploratory Data Analysis (EDA)**: Statistical profiling and cross-asset correlation mapping.
3. **Multivariate Feature Engineering**: Implementation of 10, 20, and 50-day Simple Moving Averages (SMAs), Relative Strength Index (RSI), and lag-feature mechanisms.
4. **Statistical Baseline Modeling**: Autoregressive Integrated Moving Average (SARIMA) parameter optimization.
5. **Recursive Deep Learning**: Gated Recurrent Unit (GRU) modeling for non-linear temporal dependency capture.
6. **Model Evaluation & Projection**: Comparative performance benchmarking and immediate-horizon trend forecasting.

## 1. Environment Setup and Data Acquisition

Initialization of the analytical environment and ingestion of high-resolution historical data using the `yfinance` interface.

In [ ]:
import warnings
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import pmdarima as pm

warnings.filterwarnings("ignore")
plt.style.use('fivethirtyeight')

In [ ]:
tickers = ["TCS.NS", "INFY.NS", "WIPRO.NS", "HCLTECH.NS", "TECHM.NS"]
start_date = "2008-01-01"
end_date = "2024-12-12"

stock_data = yf.download(tickers, start=start_date, end=end_date)
global_data = yf.download(['GC=F', 'INR=X', '^GSPC'], start=start_date, end=end_date)

## 2. Exploratory Data Analysis (EDA)

Systematic profiling of the constituent assets to identify sector-wide trends and establish the degree of correlation with global macroeconomic indicators (Gold, USD-INR, and S&P 500).

In [ ]:
def perform_eda(stocks, globals):
    price_col = 'Adj Close' if 'Adj Close' in stocks.columns else 'Close'
    adj_prices = stocks[price_col]
    
    # 1. Component Price Analysis
    plt.figure(figsize=(15, 6))
    for ticker in tickers:
        plt.plot(adj_prices.index, adj_prices[ticker], label=ticker, alpha=0.8)
    plt.title("Nifty 50 IT Sector Component Analysis")
    plt.legend()
    plt.show()
    
    # 2. Statistical Distribution Diagnostics
    print("--- Descriptive Statistics: Sector Components ---")
    display(adj_prices.describe())
    
    # 3. Cross-Asset Correlation Matrix
    g_prices = globals[price_col].rename(columns={'GC=F': 'Gold', 'INR=X': 'USD_INR', '^GSPC': 'SP500'})
    sector_mean = adj_prices.mean(axis=1).to_frame(name='Sector_Mean')
    correlation_df = sector_mean.join(g_prices, how='inner').corr()
    
    plt.figure(figsize=(10, 6))
    sns.heatmap(correlation_df, annot=True, cmap='RdYlGn', center=0)
    plt.title("Sector-Global Interdependency Correlation Matrix")
    plt.show()

perform_eda(stock_data, global_data)

## 3. Multivariate Feature Engineering

Calculation of the volume-weighted Aggregate Performance Score (APS) and its augmentation with momentum oscillators and structural moving averages.

In [ ]:
def build_analytical_dataset(stocks, globals):
    # 1. APS Metric Engineering
    price_col = 'Adj Close' if 'Adj Close' in stocks.columns else 'Close'
    adj_close = stocks[price_col]
    volume = stocks['Volume'].replace(0, np.nan).ffill()
    aps = (adj_close * volume).sum(axis=1) / volume.sum(axis=1)
    df = pd.DataFrame({'APS': aps})
    
    # 2. Macroeconomic Indicator Integration
    g_col = 'Adj Close' if 'Adj Close' in globals.columns else 'Close'
    gold = globals[g_col]['GC=F'].to_frame(name='Gold')
    usd_inr = globals[g_col]['INR=X'].to_frame(name='USD_INR')
    sp500 = globals[g_col]['^GSPC'].to_frame(name='SP500_INR')
    df = df.join([gold, usd_inr, sp500], how='inner')
    df['SP500_INR'] = df['SP500_INR'] * df['USD_INR']
    
    # 3. Technical Momentum & Trend Indicators
    for window in [10, 20, 50]:
        df[f'SMA_{window}'] = df['APS'].rolling(window=window).mean()
    
    delta = df['APS'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / loss
    df['RSI'] = 100 - (100 / (1 + rs))
    
    # 4. Temporal Lag Mechanisms (t-1 to t-5)
    for i in range(1, 6):
        df[f'APS_lag_{i}'] = df['APS'].shift(i)
    
    return df.dropna()

dataset = build_analytical_dataset(stock_data, global_data)
display(dataset.tail())

## 4. Comparative Model Development

Sequential partitioning of the dataset for the training and validation of a statistical baseline (SARIMA) and a multivariate recurrent neural network (GRU).

In [ ]:
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(dataset)

def create_sequences(data, time_steps=60):
    X, y = [], []
    for i in range(time_steps, len(data)):
        X.append(data[i-time_steps:i, :])
        y.append(data[i, 0])
    return np.array(X), np.array(y)

X, y = create_sequences(scaled_data)
split = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

sarima_model = pm.auto_arima(dataset['APS'].iloc[:split], seasonal=True, m=5, suppress_warnings=True)

gru_model = Sequential([
    GRU(64, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dense(1)
])
gru_model.compile(optimizer=Adam(0.001), loss='mse')
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
history = gru_model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=30, batch_size=32, callbacks=[early_stop], verbose=0)

## 5. Performance Metrics and Immediate-Horizon Projection

Evaluation of model accuracy across a standardized test set and the generation of high-resolution trend projections within the immediate temporal window.

In [ ]:
gru_preds = gru_model.predict(X_test, verbose=0)
sarima_preds = sarima_model.predict(n_periods=len(y_test))

print("--- Model Benchmarking Diagnostics ---")
print(f"GRU Coefficient of Determination (R2): {r2_score(y_test, gru_preds):.4f}")
print(f"SARIMA Coefficient of Determination (R2): {r2_score(y_test, sarima_preds if len(sarima_preds)==len(y_test) else y_test[:len(sarima_preds)]):.4f}")

def project_future_horizon(model, last_seq, steps=10):
    curr = last_seq.copy()
    preds = []
    for _ in range(steps):
        p = model.predict(curr.reshape(1, curr.shape[0], curr.shape[1]), verbose=0)
        preds.append(p[0,0])
        new_row = curr[-1].copy()
        new_row[0] = p[0,0]
        curr = np.roll(curr, -1, axis=0)
        curr[-1] = new_row
    return np.array(preds)

future_scaled = project_future_horizon(gru_model, scaled_data[-60:])
dummy = np.zeros((10, dataset.shape[1]))
dummy[:, 0] = future_scaled
project_future_aps = scaler.inverse_transform(dummy)[:, 0]

plt.figure(figsize=(12, 6))
immediate_history = dataset['APS'].iloc[-10:]
future_dates = pd.date_range(start=dataset.index[-1] + pd.Timedelta(days=1), periods=10)

plt.plot(immediate_history.index, immediate_history, label='Observed (Last 10 Days)', marker='o')
plt.plot(future_dates, project_future_aps, label='Projected Trend (Next 10 Days)', color='#e74c3c', marker='x', linestyle='--')
plt.title('Short-Term Trend Analysis: Observed vs. Projected Horizon')
plt.legend()
plt.show()